# Ore phase segmentation + intergrowth classification — experiment

Running this notebook top-to-bottom on a CUDA machine (V100) will:
1. build 5-class labels (GMM pseudo-labels + 42 hand talc masks),
2. train a SegFormer-B2 phase segmenter,
3. report **talc-fraction MAE** and **intergrowth-type accuracy**,
4. show colour overlays (green=обычные, red=тонкие, blue=тальк).

In [ ]:
import sys, pathlib, torch
sys.path.insert(0, str(pathlib.Path.cwd().parent / 'src'))
sys.path.insert(0, str(pathlib.Path.cwd() / 'src'))  # if run from repo root
from orenet import constants as C
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device, '| classes:', C.CLASS_NAMES)

# --- experiment config (tune here) ---
EPOCHS = 12
STEPS_PER_EPOCH = 250
CROP = 512
BATCH = 8
MAX_TRAIN_IMAGES = 500   # cap non-talc images cached for the first run; None = all
CACHE_DIR = pathlib.Path('data/derived/cache')

## 1. Index dataset and split by slide (no leakage)

In [ ]:
from orenet.data_index import build_index
from sklearn.model_selection import GroupShuffleSplit

records = build_index()
print('total records:', len(records))

def gsplit(recs, test_size):
    if len(recs) < 5:
        return recs, recs
    groups = [r.slide_id for r in recs]
    tr, te = next(GroupShuffleSplit(1, test_size=test_size, random_state=42).split(recs, groups=groups))
    return [recs[i] for i in tr], [recs[i] for i in te]

talc = [r for r in records if r.talc_label is not None]
rest = [r for r in records if r.talc_label is None]
assert talc, 'No talc labels found. Run first_labling_attempt/export_to_yoloseg.py first.'
# honour the hand YOLO-seg train/val split so val talc is truly held out
talc_tr = [r for r in talc if r.talc_split == 'train']
talc_ev = [r for r in talc if r.talc_split == 'val']
if not talc_ev:
    talc_tr, talc_ev = gsplit(talc, 0.2)
rest_tr, rest_ev = gsplit(rest, 0.15)
if MAX_TRAIN_IMAGES is not None:
    rest_tr = rest_tr[:MAX_TRAIN_IMAGES]
train_recs = talc_tr + rest_tr
eval_recs = talc_ev + rest_ev
print(f'train={len(train_recs)} (talc {len(talc_tr)})  eval={len(eval_recs)} (talc {len(talc_ev)})')

## 2. Build the label cache (GMM pseudo-labels + talc masks) — runs once

In [ ]:
from orenet.cache import build_cache
train_items = build_cache(train_recs, CACHE_DIR / 'train')
eval_items = build_cache(eval_recs, CACHE_DIR / 'eval')
print('cached train:', len(train_items), '| eval:', len(eval_items),
      '| talc in eval:', sum(it.has_talc for it in eval_items))

## 3. Data loaders (patch crops, rare-class sampling, talc copy-paste)

In [ ]:
from torch.utils.data import DataLoader
from orenet.dataset import PatchDataset

train_ds = PatchDataset(train_items, crop=CROP, length=STEPS_PER_EPOCH * BATCH, augment=True)
val_ds = PatchDataset(eval_items, crop=CROP, length=200, augment=False, copy_paste_p=0.0)
train_loader = DataLoader(train_ds, batch_size=BATCH, num_workers=4, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH, num_workers=2)
b = next(iter(train_loader))
print('batch image', tuple(b['image'].shape), 'label', tuple(b['label'].shape))

## 4. Model — SegFormer-B2 (ADE20k pretrained encoder)

In [ ]:
from orenet.model import SegmenterConfig, build_segmenter
model = build_segmenter(SegmenterConfig(pretrained=True, n_classes=C.NUM_CLASSES))
n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'{n_params:.1f}M params')

## 5. Train

In [ ]:
from orenet.engine import TrainConfig, train
cfg = TrainConfig(epochs=EPOCHS, steps_per_epoch=STEPS_PER_EPOCH)
history = train(model, train_loader, val_loader, device, cfg)

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(history['loss']); ax[0].set_title('train loss'); ax[0].set_xlabel('epoch')
ax[1].plot(history['miou'], label='mIoU'); ax[1].plot(history['talc_iou'], label='talc IoU')
ax[1].set_title('val IoU'); ax[1].set_xlabel('epoch'); ax[1].legend(); plt.tight_layout(); plt.show()

## 6. Required metrics: talc-fraction MAE + intergrowth-type accuracy

In [ ]:
from orenet.metrics import evaluate
res = evaluate(model, eval_items, device, C.NUM_CLASSES)
print(f'Talc-fraction MAE     : {res.talc_mae*100:.2f}%  (n={res.talc_n})')
print(f'Intergrowth accuracy  : {res.intergrowth_acc*100:.1f}%  (n={res.intergrowth_n})')

## 7. Qualitative overlays (green=обычные, red=тонкие, blue=тальк)

In [ ]:
import cv2, numpy as np, matplotlib.pyplot as plt
from orenet.inference import predict
from orenet.grains import extract_grains
from orenet.viz import make_overlay

samples = [it for it in eval_items if it.has_talc][:2] + \
          [it for it in eval_items if it.sort == 'fine'][:1] + \
          [it for it in eval_items if it.sort == 'normal'][:1]
fig, axes = plt.subplots(len(samples), 2, figsize=(11, 4 * len(samples)))
for row, it in enumerate(samples):
    img = cv2.imread(str(it.image_path), cv2.IMREAD_COLOR)
    cmap, _ = predict(model, img, device, C.NUM_CLASSES)
    grains = extract_grains(cmap)
    overlay = make_overlay(img, cmap, grains)
    axes[row, 0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); axes[row, 0].set_title(f'{it.sort}: original'); axes[row, 0].axis('off')
    axes[row, 1].imshow(overlay); axes[row, 1].set_title('overlay'); axes[row, 1].axis('off')
plt.tight_layout(); plt.show()

## 8. Save checkpoint

In [ ]:
ckpt = pathlib.Path('outputs'); ckpt.mkdir(exist_ok=True)
torch.save(model.state_dict(), ckpt / 'segformer_phases.pt')
print('saved', ckpt / 'segformer_phases.pt')